# MCP in LangChain

MCP tools get automatically converted to LangChain tools and work with agents like any other tool.

## Setup


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

## Part 1: Connect to an MCP Server

Connect to the `mcp-time` server using stdio transport.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to the mcp-time server
mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)

# Load tools from the MCP server
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

### Create an Agent with MCP Tools

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="You are a helpful assistant with access to time-related tools.",
)

### Test the Agent


In [ ]:
result = await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": "What's the time in San Francisco right now?"}
        ]
    }
)

for msg in result["messages"]:
    msg.pretty_print()

## Part 2: Connect to Your Custom MCP Server

Connect to the MCP server you built in Exercise 1. Make sure it works with the MCP Inspector before continuing.

In [ ]:
# Connect to your custom MCP server from Exercise 1
# Make sure you completed Exercise 1 before running this!
my_client = MultiServerMCPClient(
    {
        "my_server": {
            "transport": "stdio",
            "command": "uv",
            "args": ["run", "python", "exercise1 - simple-server.py"],
        }
    }
)

In [ ]:
my_tools = await my_client.get_tools()
print(f"Loaded {len(my_tools)} tools: {[t.name for t in my_tools]}")

In [ ]:
my_agent = create_agent(
    model=llm,
    tools=my_tools,
    system_prompt="You have access to a custom MCP server with greeting and math tools.",
)

In [ ]:
from langchain_core.messages import HumanMessage

result = await my_agent.ainvoke(
    {"messages": [HumanMessage(content="Say hello to Alice and then add 5 + 3")]}
)
result["messages"][-1].pretty_print()

## Try It Yourself

### Exercise: Combine Multiple MCP Servers

Connect to both the time server and your custom server at the same time. Then ask a question that uses tools from both servers.

**Hint:** `MultiServerMCPClient` accepts multiple server configs in the same dictionary.

In [ ]:
# TODO: Create a MultiServerMCPClient with both the "time" server and your custom server
# Hint: Pass both server configs in the same dictionary to MultiServerMCPClient

combined_client = ...

# TODO: Load all tools from both servers and print the full list
all_tools = ...

# TODO: Create an agent with all tools and an appropriate system prompt
combined_agent = ...

# TODO: Invoke the combined agent with a query that requires tools from both servers
# Try: "Say hello to Alice, then tell me what time it is in Tokyo"

## Bonus: Try Public MCP Servers

### HTTP/SSE Servers (Remote - no install needed)

Use `"transport": "sse"` and provide the `"url"` key.

| Server | URL | Description |
|---|---|---|
| Cloudflare Docs | `https://docs.mcp.cloudflare.com/sse` | Search Cloudflare documentation |
| DeepWiki | `https://mcp.deepwiki.com/sse` | Knowledge retrieval |
| Hugging Face | `https://hf.co/mcp` | ML model hub |

```python
MultiServerMCPClient({
    "cloudflare": {
        "transport": "sse",
        "url": "https://docs.mcp.cloudflare.com/sse",
    }
})
```

### npx Servers (Local - no API key needed)

| Server | npx command | Description |
|---|---|---|
| Everything | `npx -y @modelcontextprotocol/server-everything` | Official demo: `echo`, `add`, and more |
| Calculator | `npx -y @wrtnlabs/calculator-mcp@latest` | Math: add, sub, mul, div, sqrt |
| DuckDuckGo | `npx -y duckduckgo-mcp-server` | Web search, no API key |
| Weather | `npx -y @dangahagan/weather-mcp@latest` | Global weather forecasts |